# Exploration : salaires des développeurs en France

Enquête Stack Overflow 2026. Objectif : trouver les constats les plus nets
pour le message du tableau de bord destiné à la direction de la rémunération d'une ESN.

In [ ]:
import pandas as pd

from salaires.chargement import RACINE
from salaires.logique import SEUIL_N
from salaires.population import ajouter_niveau

## Chargement

Je relis le CSV brut directement avec pandas, en ne gardant que les colonnes utiles.

In [ ]:
COLONNES = ["Country", "CodeMain", "Employment", "Currency", "Compensation",
            "WorkExp", "Industry", "WorkSetting"]
brut = pd.read_csv(RACINE / "data" / "raw" / "2026" / "survey_results_public.csv", usecols=COLONNES)
brut.shape

## Population

Filtres décrits dans `src/salaires/population.py`, appliqués un par un
pour voir combien de répondants chacun retire.

In [ ]:
filtres = {
    "France": brut["Country"] == "France",
    "Code comme activité principale": brut["CodeMain"] == "I create code as a primary part of my job",
    "Salarié": brut["Employment"] == "Employed",
    "Salaire déclaré en euros": brut["Currency"] == "EUR European Euro",
    "Salaire entre 15 000 et 200 000 €": brut["Compensation"].between(15_000, 200_000),
}

masque = pd.Series(True, index=brut.index)
etapes = [("Tous les répondants", len(brut))]
for nom, condition in filtres.items():
    masque = masque & condition
    etapes.append((nom, int(masque.sum())))

pop = brut[masque].rename(columns={"Compensation": "SalaireEUR"})
pop = ajouter_niveau(pop)
pd.DataFrame(etapes, columns=["Filtre", "Répondants restants"])

## Salaire : médiane et quartiles

In [ ]:
print("Répondants :", len(pop))
pop["SalaireEUR"].quantile([0.25, 0.5, 0.75])

In [ ]:
def mediane_par(df, colonne):
    """Salaire médian et effectif pour chaque modalité de colonne."""
    tableau = df.groupby(colonne, observed=True)["SalaireEUR"].agg(mediane="median", n="count")
    tableau["fiable"] = tableau["n"] >= SEUIL_N
    return tableau

## Par niveau d'expérience (WorkExp)

Junior : 0 à 4 ans, Confirmé : 5 à 9 ans, Senior : 10 ans et plus.
Avec « moins de 3 ans », il n'y aurait que 22 juniors, sous le seuil de 30.

In [ ]:
par_niveau = mediane_par(pop, "Niveau")
print("Sans WorkExp, niveau vide :", pop["Niveau"].isna().sum())
par_niveau

In [ ]:
ecart = par_niveau.loc["Senior", "mediane"] / par_niveau.loc["Junior", "mediane"] - 1
print(f"Senior par rapport à Junior : +{ecart:.0%}")

## Par secteur (Industry)

Seuls deux secteurs dépassent le seuil de 30 répondants : les ESN et les éditeurs de logiciels.

In [ ]:
par_secteur = mediane_par(pop, "Industry").sort_values("n", ascending=False)
par_secteur

In [ ]:
ESN = "IT services, consulting, systems integration, or managed services"
EDITEUR = "Software development"

esn = par_secteur.loc[ESN, "mediane"]
editeur = par_secteur.loc[EDITEUR, "mediane"]
print(f"ESN : {esn:.0f} €, éditeurs : {editeur:.0f} €, écart : {esn / editeur - 1:.0%}")

# L'écart vient-il d'une différence d'expérience ? Expérience médiane dans chaque secteur :
pop[pop["Industry"].isin([ESN, EDITEUR])].groupby("Industry")["WorkExp"].median()

## Par mode de travail (WorkSetting)

In [ ]:
par_mode = mediane_par(pop, "WorkSetting").sort_values("mediane")
par_mode

In [ ]:
# Les modes de travail n'ont pas la même expérience : le présentiel compte beaucoup de juniors.
pd.crosstab(pop["WorkSetting"], pop["Niveau"])

## Constats retenus pour le message

1. **Le salaire suit l'expérience** : médiane de 38 315 € pour les juniors (n = 56),
   48 000 € pour les confirmés (n = 107), 62 000 € pour les seniors (n = 207), soit +62 %.
2. **Les ESN paient moins que les éditeurs** : 44 500 € (n = 66) contre 53 000 € (n = 65),
   soit 16 % de moins, à expérience médiane comparable (10,5 ans contre 10).
3. **Le télétravail complet est le mieux payé** : 65 000 € en télétravail (n = 65)
   contre 41 394 € en présentiel (n = 44). Écart en partie dû à l'expérience :
   45 % des répondants en présentiel sont juniors, contre 6 % en télétravail.